# `Устанавливаем необходимые зависимости`

In [41]:
! pip3 install -q pyspark pyarrow parquet-tools

# `Готовим SparkContext`

Oбъект SparkContext является точкой входа для работы со Spark-кластером.

In [38]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext

In [44]:
# Создаём конфигурационный класс с параметрами подключения
conf = (
    SparkConf()
        # Указываем URL master ноды Spark кластера
        # Можно использовать local mode, указав `local[<number_cores>]`
        # В таком случае вся обработка будет происходить на текущем компьютере
        # При этом, это может давать преимущество ввиду наличия параллелизма по ядрам компьютера
        .setMaster('local[*]')
)

# Создаём точку доступа на кластер. Позволяет использовать RDD API
sc = SparkContext(conf=conf)

# Точка доступа для использования DataFrame API
spark = SparkSession(sc)

# По завершении программы нужно обязательно выполнить остановку подключения для освобождения занятых ресурсов
# sc.stop()

# `Загрузка данных`

In [45]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet

--2026-03-20 11:02:04--  https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet [following]
--2026-03-20 11:02:04--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24369 (24K) [application/octet-stream]
Saving to: ‘calendar.parquet.1’

calendar.parquet.1  100%[===================>]  23.80K  --.-KB/s    in 0.001s  

2026-03-20 11:02:05 (21.9 MB/s) - ‘calendar.parquet.1’ saved [24369/24369]



Атрибуты датасета **calendar.parquet**.

Датасет содержит календарные данные, которые помогают связать продажи товаров с конкретными днями, неделями и событиями.

1. **date**: Дата записи данных.

2. **wm_yr_wk**: Календарная неделя в формате года и номера недели (год-неделя).

3. **weekday**: Название дня недели.

4. **wday**: Порядковый номер дня недели.

5. **month**: Порядковый номер месяца.

6. **year**: Год наблюдения.

7. **d**: Идентификатор дня в формате последовательности (например, d_1, d_2 и т.д.).

8. **event_name_1**: Название первичного события (праздники, акции, особые события).

9. **event_type_1**: Тип первичного события (например, праздник, спортивное событие).

10. **event_name_2**: Название вторичного события (если в этот день несколько значимых событий).

11. **event_type_2**: Тип вторичного события.

12. **snap_CA, snap_TX, snap_WI**: Индикаторы (0 или 1) программы SNAP (льготная покупка продуктов) в соответствующих штатах (Калифорния, Техас, Висконсин). Показатель 1 означает, что в указанный день были доступны льготы SNAP.

In [46]:
# Считаем файл calendar.parquet и запишем данные в DataFrame с названием df_calendar
df_calendar = spark.read.parquet("calendar.parquet")
df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|
|2011-02-02|   11101|Wednesday|   5|    2|2011|d_5|        NULL|        NULL|        NULL|        NULL| 

In [47]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet

--2026-03-20 11:02:32--  https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet [following]
--2026-03-20 11:02:32--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31179351 (30M) [application/octet-stream]
Saving to: ‘sales.parquet.1’

sales.parquet.1     100%[===================>]  29.73M   188MB/s    in 0.2s    

2026-03-20 11:02:33 (188 MB/s) - ‘sales.parquet.1’ saved [31179351/31179351]



Атрибуты датасета **sales.parquet**.

Данный датасет содержит историю продаж товаров в розничных магазинах Walmart. Используется для анализа спроса, прогнозирования продаж и оптимизации запасов.

1. **id**: Уникальный идентификатор товара в конкретном магазине.

2. **item_id**: Идентификатор товара.

3. **dept_id**: Идентификатор отдела, к которому относится товар.

4. **cat_id**: Категория товара.

5. **store_id**: Идентификатор магазина, в котором был продан товар.

6. **state_id**: Штат, в котором расположен магазин.

7. **d_1, d_2, ..., d_n**: Ежедневные данные о продажах данного товара в указанном магазине (количество проданных единиц за каждый день). Каждый атрибут соответствует одному дню, начиная с первого дня периода наблюдения.

In [48]:
# Считаем файл sales.parquet и запишем данные в DataFrame с названием df_sales
df_sales = spark.read.parquet("sales.parquet")
df_sales.show(5)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [49]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet

--2026-03-20 11:02:40--  https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet [following]
--2026-03-20 11:02:41--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2385868 (2.3M) [application/octet-stream]
Saving to: ‘prices.parquet.1’

prices.parquet.1    100%[===================>]   2.27M  --.-KB/s    in 0.06s   

2026-03-20 11:02:41 (35.5 MB/s) - ‘prices.parquet.1’ saved [2385868/2385868]



Атрибуты датасета **prices.parquet**.

Датасет содержит информацию о динамике цен на товары в различных магазинах за определенные недели.

1. **store_id** – идентификатор магазина, в котором продаётся товар.

2. **item_id** – уникальный идентификатор конкретного товара.

3. **wm_yr_wk** – календарная неделя года, к которой относится указанная цена (в формате Walmart-календаря).

4. **sell_price** – розничная цена продажи товара в указанном магазине в течение соответствующей недели.

In [50]:
# Считаем файл prices.parquet и запишем данные в DataFrame с названием df_prices
df_prices = spark.read.parquet("prices.parquet") # ВАШ КОД
df_prices.show(5)

+--------+-------------+--------+----------+
|store_id|      item_id|wm_yr_wk|sell_price|
+--------+-------------+--------+----------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|
|    CA_1|HOBBIES_1_001|   11326|      9.58|
|    CA_1|HOBBIES_1_001|   11327|      8.26|
|    CA_1|HOBBIES_1_001|   11328|      8.26|
|    CA_1|HOBBIES_1_001|   11329|      8.26|
+--------+-------------+--------+----------+
only showing top 5 rows


# `Задачи DataFrame API`

In [51]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [52]:
# 1. Создайте новую колонку 'is_weekend' в df_calendar, которая показывает выходной день (Saturday or Saturday - 1, в ином случае 0).
# Используйте только DataFrame API и функции, импортированные из F.
# Подсказка: Используйте функции: when, isin.
df_calendar = df_calendar.withColumn('is_weekend', F.when(F.col('weekday').isin(['Saturday','Sunday']), 1).otherwise(0))
df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|   11101|Wednes

In [53]:
# 2. Переименуйте колонку 'wm_yr_wk' в 'week_id' в df_calendar
df_calendar = df_calendar.withColumnRenamed("wm_yr_wk","week_id") # ВАШ КОД
df_calendar.show(5)

+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|week_id|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|  11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|  11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|  11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|  11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|  11101|Wednesday|   5

In [54]:
# 3. Выведите уникальные значения weekday в df_calendar
df_calendar.createOrReplaceTempView("df_calendar")
spark.sql("SELECT DISTINCT weekday FROM df_calendar").show()

+---------+
|  weekday|
+---------+
|Wednesday|
|  Tuesday|
|   Friday|
| Thursday|
| Saturday|
|   Monday|
|   Sunday|
+---------+



In [55]:
# 4. Выведите количество уникальных магазинов в df_sales
# Подсказка: используйте атрибут 'store_id'
df_sales.createOrReplaceTempView("df_sales")
spark.sql("select count(DISTINCT store_id) as count_unique from df_sales").show()

+------------+
|count_unique|
+------------+
|          10|
+------------+



In [56]:
# 5. Выведите среднее количество продаж по категориям товаров за d_1
# Подсказка: используйте датасет df_sales
spark.sql("select cat_id, count(*) from df_sales group by cat_id").show()

+---------+--------+
|   cat_id|count(1)|
+---------+--------+
|    FOODS|   14370|
|HOUSEHOLD|   10470|
|  HOBBIES|    5650|
+---------+--------+



In [57]:
df_sales.show(5)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [58]:
# 6. Создайте новый атрибут 'sell_price_int' в df_prices, изменив тип данных атрибута 'sell_price' на Integer
# Подсказка, для изменения типа используйте метод cast
df_prices = df_prices.withColumn("sell_price_int", F.col("sell_price").cast("Integer"))
df_prices.show(5)

+--------+-------------+--------+----------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+-------------+--------+----------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|
+--------+-------------+--------+----------+--------------+
only showing top 5 rows


In [59]:
# 7. Выведите 5 самых дорогих товаров в df_prices
df_prices.createOrReplaceTempView("df_prices")
spark.sql("select item_id, sell_price from df_prices order by sell_price desc limit 5").show()

+---------------+----------+
|        item_id|sell_price|
+---------------+----------+
|HOUSEHOLD_2_406|    107.32|
|HOUSEHOLD_2_406|    107.32|
|HOUSEHOLD_2_406|    107.32|
|HOUSEHOLD_2_406|     61.46|
|HOUSEHOLD_2_406|     61.46|
+---------------+----------+



In [60]:
# 8. Используя sql, найдите среднее количество продаж по штатам (назвав атрибут 'avg_sales') для d1
df_sales.createOrReplaceTempView("df_sales")
spark.sql("select state_id, avg(d_1) as avg_sales from df_sales group by state_id").show(5)

+--------+------------------+
|state_id|         avg_sales|
+--------+------------------+
|      CA|1.1639061987536898|
|      TX|1.0318137094129223|
|      WI|0.9837105061768886|
+--------+------------------+



In [61]:
# 9. Выведите 10 строк датасета с товарами, проданных в Техасе
# Подсказка: используйте датасет df_sales
spark.sql("select * from df_sales where state_id='TX'").show(10)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [62]:
# 10. Выведите 5 самых высоких цен на товары (sell_price), которые были в период с "2016-01-01" по "2016-01-31"
# Подсказка: используйте join, используйте метод between, используйте метод distinct() для уникальности цен
two = (
    df_prices.join(df_calendar, df_prices.wm_yr_wk == df_calendar.week_id, how="inner")
    .filter(F.col("date").between("2016-01-01", "2016-01-31"))
    .select("sell_price")
    .distinct()
    .orderBy(F.col("sell_price").desc())
    .limit(5)
)

two.show()

+----------+
|sell_price|
+----------+
|     29.97|
|     29.96|
|     28.96|
|     27.98|
|     26.98|
+----------+



In [63]:
df_prices.show(5)

+--------+-------------+--------+----------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+-------------+--------+----------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|
+--------+-------------+--------+----------+--------------+
only showing top 5 rows


In [64]:
df_calendar.show(5)

+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|week_id|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|  11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|  11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|  11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|  11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|  11101|Wednesday|   5

In [65]:
# 11. Найдите товары с наибольшими продажами в каждой категории (cat_id) за d_1, используя датасет df_sales
# Подсказка: используйте оконную функцию rank()
# ВАШ КОД

In [66]:
# 12. Создайте новую колонку "price_category" в df_prices, указав высокая или низкая цена: если цена больше 5, то "High", иначе "Low"
# Подсказка: используйте конструкцию when().otherwise
df_prices = df_prices.withColumn('price_category', F.when(F.col('sell_price') > 5, "High").otherwise("Low"))


In [67]:
df_prices.show(15)

+--------+-------------+--------+----------+--------------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|price_category|
+--------+-------------+--------+----------+--------------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11330|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11331|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11332|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11333|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11334|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001| 

In [75]:
# 13. Вычислите суммарные продажи по всем магазинам (store_id) за d_1
df_sales.groupBy('store_id').sum('d_1').show(100)

+--------+--------+
|store_id|sum(d_1)|
+--------+--------+
|    TX_2|    3852|
|    TX_1|    2556|
|    CA_4|    1625|
|    CA_2|    3494|
|    CA_1|    4337|
|    CA_3|    4739|
|    WI_2|    2256|
|    WI_3|    4038|
|    WI_1|    2704|
|    TX_3|    3030|
+--------+--------+



In [70]:
df_sales.show(5)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

# `Задачи RDD`

### Данные

Файл - `transaction.csv`

Формат записей:

```
user_id, timestamp, item_id, category, price, quantity, city
```





In [86]:
# Задание 0. Загрузите данные в RDD
rdd = spark.sparkContext.textFile("/content/transactions.csv")

rdd.take(1)

['user_id,timestamp,item_id,category,price,quantity,city']

In [87]:
rdd.take(4)

['user_id,timestamp,item_id,category,price,quantity,city',
 '22,2025-03-07 15:41,E22,beauty,1227,1,Perm',
 '45,2025-03-19 20:27,Q77,auto,98,1,Kazan',
 '37,2025-03-04 14:55,B07,books,1435,1,Rostov']

In [88]:
perva = rdd.first()
data_rdd = rdd.filter(lambda line: line != perva)
category_price_qty_rdd = data_rdd.map(lambda line: line.split(',')).map(lambda cols: (cols[3], float(cols[4]), float(cols[5])))

In [89]:
category_price_qty_rdd.take(5)

[('beauty', 1227.0, 1.0),
 ('auto', 98.0, 1.0),
 ('books', 1435.0, 1.0),
 ('auto', 630.0, 1.0),
 ('beauty', 1248.0, 4.0)]

In [91]:

# Задание 1. Найти ТОП-5 категорий по общей выручке
category_revenue = category_price_qty_rdd.map(lambda x: (x[0], x[1] * x[2])).reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False)
top5 = category_revenue.take(5)

for x in top5:
    print(x)


('beauty', 453070.0)
('home', 444245.0)
('kids', 403922.0)
('sport', 389754.0)
('toys', 381717.0)


In [92]:
# Останавливаем существующую сессию и контекст
try:
    spark.stop()
except:
    pass

try:
    sc.stop()
except:
    pass

In [ ]:


# Задание 2. Найти пользователей, покупавших в более чем двух категориях

user_cat = ... # ВАШ КОД

... # ВАШ КОД

users_many = ... # ВАШ КОД

for u in users_many.take(10):
    print(u[0], " → ", len(u[1]), "категорий:", u[1])




In [ ]:
# Задание 3. Найти средний чек по каждому городу

city_pairs = ... # ВАШ КОД

... # ВАШ КОД

sorted_avg = ... # ВАШ КОД

for x in sorted_avg.collect():
    print(x)




In [ ]:
# Задание 4. Найти самый продаваемый товар внутри каждой категории и
# суммарное количество проданных единиц этого товара

cat_item = ... # ВАШ КОД

... # ВАШ КОД

top_items = ... # ВАШ КОД

for x in top_items.collect():
    print(x)




In [ ]:
# Задание 5. Определите час суток с максимальной выручкой.
# Найдите выручку за этот час и определите ее долю в общей выручке.
# Подсказка: создайте и используйте функцию для парсинга часа из поля timestamp

from datetime import datetime

def extract_hour(ts):
    ... # ВАШ КОД

hour_rev = ... # ВАШ КОД

... # ВАШ КОД
peak_hour = ... # ВАШ КОД
total_revenue = ... # ВАШ КОД
peak_share = ... # ВАШ КОД

print("Час пик:", peak_hour)
print("Доля общего оборота:", peak_share)